# TP2 — SPLN 2025/26
Pipeline IR + QA sobre artigos Wikipedia dos países do mundo.

**Secções:** 1. Corpus · 2. Pré-processamento · 3. Retriever · 4. QA Extractivo · 5. QA Abstractivo · 6. Demo

## 1. Corpus

In [108]:
import os
if os.path.exists("data/embeddings.npy"):
    os.remove("data/embeddings.npy")
    print("Cache de embeddings apagado")

In [109]:
# Instalar (uma vez):
! pip install wikipedia-api tqdm rank-bm25 sentence-transformers transformers datasets accelerate evaluate

import re, time, json, unicodedata
from pathlib import Path
import wikipediaapi
from tqdm.auto import tqdm

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [110]:
COUNTRIES = [
    "Afghanistan", "Albania", "Algeria", "Andorra", "Angola",
    "Antigua and Barbuda", "Argentina", "Armenia", "Australia", "Austria",
    "Azerbaijan", "Bahamas", "Bahrain", "Bangladesh", "Barbados",
    "Belarus", "Belgium", "Belize", "Benin", "Bhutan",
    "Bolivia", "Bosnia and Herzegovina", "Botswana", "Brazil", "Brunei",
    "Bulgaria", "Burkina Faso", "Burundi", "Cambodia", "Cameroon",
    "Canada", "Cape Verde", "Central African Republic", "Chad", "Chile",
    "China", "Colombia", "Comoros", "Costa Rica", "Croatia",
    "Cuba", "Cyprus", "Czech Republic", "Denmark", "Djibouti",
    "Dominica", "Dominican Republic", "East Timor", "Ecuador", "Egypt",
    "El Salvador", "Equatorial Guinea", "Eritrea", "Estonia", "Eswatini",
    "Ethiopia", "Fiji", "Finland", "France", "Gabon",
    "Gambia", "Germany", "Ghana", "Greece",
    "Grenada", "Guatemala", "Guinea", "Guinea-Bissau", "Guyana",
    "Haiti", "Honduras", "Hungary", "Iceland", "India",
    "Indonesia", "Iran", "Iraq", "Ireland", "Israel",
    "Italy", "Ivory Coast", "Jamaica", "Japan", "Jordan",
    "Kazakhstan", "Kenya", "Kiribati", "Kuwait", "Kyrgyzstan",
    "Laos", "Latvia", "Lebanon", "Lesotho", "Liberia",
    "Libya", "Liechtenstein", "Lithuania", "Luxembourg", "Madagascar",
    "Malawi", "Malaysia", "Maldives", "Mali", "Malta",
    "Marshall Islands", "Mauritania", "Mauritius", "Mexico", "Micronesia",
    "Moldova", "Monaco", "Mongolia", "Montenegro", "Morocco",
    "Mozambique", "Myanmar", "Namibia", "Nauru", "Nepal",
    "Netherlands", "New Zealand", "Nicaragua", "Niger", "Nigeria",
    "North Korea", "North Macedonia", "Norway", "Oman", "Pakistan",
    "Palau", "Panama", "Papua New Guinea", "Paraguay", "Peru",
    "Philippines", "Poland", "Portugal", "Qatar", "Romania",
    "Russia", "Rwanda", "Saint Kitts and Nevis", "Saint Lucia",
    "Saint Vincent and the Grenadines", "Samoa", "San Marino",
    "Sao Tome and Principe", "Saudi Arabia", "Senegal",
    "Serbia", "Seychelles", "Sierra Leone", "Singapore", "Slovakia",
    "Slovenia", "Solomon Islands", "Somalia", "South Africa", "South Korea",
    "South Sudan", "Spain", "Sri Lanka", "Sudan", "Suriname",
    "Sweden", "Switzerland", "Syria", "Tajikistan", "Tanzania",
    "Thailand", "Togo", "Tonga", "Trinidad and Tobago", "Tunisia",
    "Turkey", "Turkmenistan", "Tuvalu", "Uganda", "Ukraine",
    "United Arab Emirates", "United Kingdom", "United States", "Uruguay",
    "Uzbekistan", "Vanuatu", "Vatican City", "Venezuela", "Vietnam",
    "Yemen", "Zambia", "Zimbabwe",
    "Democratic Republic of the Congo", "Republic of the Congo",
    "Georgia (country)",
]

print(f"{len(COUNTRIES)} países")

194 países


In [111]:
wiki = wikipediaapi.Wikipedia(
    user_agent="SPLN-TP2-UMinho (student@uminho.pt)",
    language="en",
)

SKIP = {"See also", "References", "External links", "Notes",
        "Further reading", "Bibliography", "Footnotes", "Citations"}

def slugify(name):
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_").lower()

def sections_text(section):
    parts = []
    for sub in section.sections:
        if sub.title in SKIP:
            continue
        if sub.text.strip():
            parts.append(sub.text.strip())
        parts.extend(sections_text(sub))
    return parts

failed = []
for country in tqdm(COUNTRIES):
    path = DATA_DIR / f"{slugify(country)}.txt"
    if path.exists():
        continue
    page = wiki.page(country)
    if not page.exists():
        failed.append(country)
        continue
    parts = [page.summary] + sections_text(page)
    text = "\n\n".join(p for p in parts if p)
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    path.write_text(text, encoding="utf-8")
    time.sleep(0.1)

print(f"Descarregados: {len(list(DATA_DIR.glob('*.txt')))}")
if failed:
    print(f"Falhas: {failed}")

  0%|          | 0/194 [00:00<?, ?it/s]

Descarregados: 194


## 2. Pré-processamento e Chunking

Lê os `.txt` da pasta `data/`, divide cada artigo em parágrafos e guarda tudo num `corpus.json`.

In [112]:
def clean_corpus_text(text):
    text = re.sub(r'\[[\w\d]{1,5}\]\.?', '', text)
    text = re.sub(r'\(\d{1,2}\)\.?', '', text)
    text = re.sub(r'\([A-Za-z]\)\.?', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def chunk_article(text, min_words=30):
    # Split PRIMEIRO (pelos \n\n originais), limpar DEPOIS cada parágrafo
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    cleaned = [clean_corpus_text(p) for p in paragraphs]
    return [p for p in cleaned if len(p.split()) >= min_words]

corpus = []
chunk_id = 0

for txt_file in sorted(DATA_DIR.glob("*.txt")):
    country = txt_file.stem.replace("_", " ").title()
    text = txt_file.read_text(encoding="utf-8")
    for para in chunk_article(text):
        corpus.append({"id": chunk_id, "country": country, "text": para})
        chunk_id += 1

corpus_path = DATA_DIR / "corpus.json"
with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)

print(f"Total chunks:     {len(corpus)}")
print(f"Países:           {len(set(c['country'] for c in corpus))}")
print(f"Média chunks/país: {len(corpus) / len(set(c['country'] for c in corpus)):.1f}")
print(f"Guardado em:      {corpus_path}")

Total chunks:     8919
Países:           194
Média chunks/país: 46.0
Guardado em:      data/corpus.json


In [113]:
# Estatísticas dos chunks
import statistics, random
word_counts = [len(c["text"].split()) for c in corpus]
print(f"Palavras por chunk — min: {min(word_counts)}, max: {max(word_counts)}, média: {statistics.mean(word_counts):.0f}, mediana: {statistics.median(word_counts):.0f}")

sample = random.choice(corpus)
print(f"\n--- Chunk #{sample['id']} ({sample['country']}) ---")
print(sample["text"][:500] + ("..." if len(sample["text"]) > 500 else ""))

Palavras por chunk — min: 30, max: 1326, média: 230, mediana: 202

--- Chunk #1824 (Cyprus) ---
At the end of the Bronze Age, the island experienced two waves of Greek settlement. The first wave consisted of Mycenaean Greek traders, who started visiting Cyprus around 1400 BC. A major wave of Greek settlement is believed to have taken place following the Late Bronze Age collapse of Mycenaean Greece from 1100 to 1050 BC, with the island's predominantly Greek character dating from this period. Cyprus occupies an important role in Greek mythology, being the birthplace of Aphrodite and Adonis, ...


## 3. Retriever Híbrido (BM25 + Sentence-BERT + RRF)

- **BM25** — léxico, com `rank-bm25`.
- **Sentence-BERT** — semântico, com modelo `multi-qa-MiniLM-L6-cos-v1`.
- **Híbrido** — Reciprocal Rank Fusion das duas rankings.

In [114]:
# Carregar corpus do disco (assim esta secção é independente)
with open(DATA_DIR / "corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

texts = [c["text"] for c in corpus]
print(f"Corpus carregado: {len(corpus)} chunks")

Corpus carregado: 8919 chunks


In [115]:
# --- BM25 (léxico) ---
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [tokenize(t) for t in texts]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 indexado ({len(tokenized_corpus)} docs)")

def bm25_search(query, top_k=10):
    scores = bm25.get_scores(tokenize(query))
    top_idx = scores.argsort()[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]

BM25 indexado (8919 docs)


In [116]:
# --- Sentence-BERT (semântico) ---
import numpy as np
from sentence_transformers import SentenceTransformer

EMB_PATH = DATA_DIR / "embeddings.npy"
MODEL_NAME = "multi-qa-MiniLM-L6-cos-v1"

st_model = SentenceTransformer(MODEL_NAME)

if EMB_PATH.exists():
    embeddings = np.load(EMB_PATH)
    print(f"Embeddings carregados do cache: {embeddings.shape}")
else:
    print("A calcular embeddings (uma vez)...")
    embeddings = st_model.encode(texts, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    np.save(EMB_PATH, embeddings)
    print(f"Guardados em {EMB_PATH} — shape {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


A calcular embeddings (uma vez)...


Batches:   0%|          | 0/279 [00:00<?, ?it/s]

Guardados em data/embeddings.npy — shape (8919, 384)


In [117]:
def semantic_search(query, top_k=10):
    q_emb = st_model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
    scores = embeddings @ q_emb
    top_idx = scores.argsort()[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]

In [118]:
# --- Híbrido (Reciprocal Rank Fusion) ---
def hybrid_search(query, top_k=10, k_rrf=60, candidates_per_retriever=50):
    bm25_results = bm25_search(query, top_k=candidates_per_retriever)
    sem_results  = semantic_search(query, top_k=candidates_per_retriever)

    rrf_scores = {}
    for ranking in [bm25_results, sem_results]:
        for rank, (doc_id, _) in enumerate(ranking):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k_rrf + rank + 1)

    ranked = sorted(rrf_scores.items(), key=lambda x: -x[1])
    return [(doc_id, score) for doc_id, score in ranked[:top_k]]

In [119]:
# --- Teste comparativo ---
def show_results(label, results, n=3):
    print(f"\n=== {label} ===")
    for doc_id, score in results[:n]:
        c = corpus[doc_id]
        snippet = c["text"][:200].replace("\n", " ")
        print(f"  [{c['country']}] score={score:.4f}")
        print(f"    {snippet}...")

test_queries = [
    "What is the capital of Portugal?",
    "Which countries border Brazil?",
    "When did Germany reunify?",
]

for q in test_queries:
    print(f"\n{'='*70}\nQUERY: {q}\n{'='*70}")
    show_results("BM25",     bm25_search(q,     top_k=3))
    show_results("Semantic", semantic_search(q, top_k=3))
    show_results("Hybrid",   hybrid_search(q,   top_k=3))


QUERY: What is the capital of Portugal?

=== BM25 ===
  [Portugal] score=23.1998
    The region has been inhabited by humans since approximately 400,000 years ago. Neanderthals roamed the northern Iberian peninsula, and a hominin tooth has been found at the Nova da Columbeira Cave in ...
  [Portugal] score=22.7335
    Portugal has a population of 10,749,635, of whom 9,205,938 are Portuguese nationals and the remainder are foreign residents at 1,543,697, as of 2024. Portugal is steadily aging and has the world's thi...
  [Portugal] score=22.2331
    Portugal, officially the Portuguese Republic, is a country in Southwestern Europe. It is a unitary republic made up by mainland Portugal and two autonomous regions, with Lisbon as its capital and larg...

=== Semantic ===
  [Portugal] score=0.6250
    Portugal, officially the Portuguese Republic, is a country in Southwestern Europe. It is a unitary republic made up by mainland Portugal and two autonomous regions, with Lisbon as its capital 

## 4. QA Extractivo — Fine-tuning DistilBERT em SQuAD

> ⚡ **Esta secção precisa de GPU.** Recomendado: abrir no Colab com runtime T4 GPU.
>
> Workflow Colab:
> 1. Upload notebook + pasta `data/` para Colab (barra lateral esquerda 📁)
> 2. `Runtime` → `Change runtime type` → **T4 GPU**
> 3. Run All — sec. 1-2 fazem cache hit, sec. 3 carrega embeddings, sec. 4 treina (~20-40 min)
> 4. O modelo treinado é guardado em `/content/drive/MyDrive/SPLN/TP2/models/` (Drive) para persistir entre sessões.

In [120]:
# Setup do ambiente
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = Path("/content/drive/MyDrive/SPLN/TP2")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    MODELS_DIR = DRIVE_BASE / "models"
    os.system("pip install -q transformers datasets accelerate evaluate")
else:
    MODELS_DIR = Path("models")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Modelos vão para: {MODELS_DIR}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Sem GPU — fine-tuning vai demorar HORAS. Considera Colab.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelos vão para: /content/drive/MyDrive/SPLN/TP2/models
PyTorch: 2.10.0+cu128
CUDA disponível: True
GPU: Tesla T4


In [121]:
# Carregar SQuAD v1.1 + subset
from datasets import load_dataset

squad = load_dataset("squad")
print(squad)

TRAIN_SIZE = 20000
EVAL_SIZE = 500

train_subset = squad["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
eval_subset  = squad["validation"].shuffle(seed=42).select(range(EVAL_SIZE))

print(f"\nTrain subset: {len(train_subset)} exemplos")
print(f"Eval subset:  {len(eval_subset)} exemplos")
print(f"\nExemplo:")
ex = train_subset[0]
print(f"  Q: {ex['question']}")
print(f"  Context: {ex['context'][:200]}...")
print(f"  A: {ex['answers']['text']}")

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Train subset: 20000 exemplos
Eval subset:  500 exemplos

Exemplo:
  Q: What percentage of Egyptians polled support death penalty for those leaving Islam?
  Context: The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan indepen...
  A: ['84%']


In [122]:
# Tokenização (padrão HuggingFace para SQuAD QA)
from transformers import AutoTokenizer

QA_MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 384
STRIDE = 128

qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)

def preprocess_training(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = qa_tokenizer(
        questions, examples["context"],
        max_length=MAX_LEN, truncation="only_second", stride=STRIDE,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding="max_length",
    )
    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions, end_positions = [], []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

train_tokenized = train_subset.map(
    preprocess_training, batched=True,
    remove_columns=train_subset.column_names
)
print(f"Train tokenizado: {len(train_tokenized)} features")

Train tokenizado: 20196 features


In [123]:
# Fine-tuning
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)
OUTPUT_DIR = MODELS_DIR / "distilbert-squad-checkpoint"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=3e-5,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(
    model=qa_model,
    args=training_args,
    train_dataset=train_tokenized,
)

trainer.train()

FINAL_MODEL_DIR = MODELS_DIR / "distilbert-squad-final"
trainer.save_model(str(FINAL_MODEL_DIR))
qa_tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
print(f"\n✅ Modelo guardado em: {FINAL_MODEL_DIR}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,3.572754
400,2.096198
600,1.861180
800,1.692979
1000,1.596010
1200,1.503827
1400,1.275849
1600,1.177192
1800,1.185051
2000,1.168869


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Modelo guardado em: /content/drive/MyDrive/SPLN/TP2/models/distilbert-squad-final


In [124]:
# Avaliação: EM e F1 no eval_subset
from transformers import pipeline
import evaluate

qa_pipe = pipeline(
    "question-answering",
    model=str(FINAL_MODEL_DIR),
    tokenizer=str(FINAL_MODEL_DIR),
    device=0 if torch.cuda.is_available() else -1,
)

squad_metric = evaluate.load("squad")

predictions = []
references = []
for ex in tqdm(eval_subset, desc="Avaliação"):
    pred = qa_pipe(question=ex["question"], context=ex["context"])
    predictions.append({"id": ex["id"], "prediction_text": pred["answer"]})
    references.append({"id": ex["id"], "answers": ex["answers"]})

results = squad_metric.compute(predictions=predictions, references=references)
print(f"\n=== Resultados no SQuAD validation ({len(eval_subset)} ex) ===")
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 score:    {results['f1']:.2f}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Avaliação:   0%|          | 0/500 [00:00<?, ?it/s]


=== Resultados no SQuAD validation (500 ex) ===
Exact Match: 69.80
F1 score:    78.52


In [125]:
# Inferência: retriever (top-k) → QA extractivo → melhor resposta
def extractive_qa(query, top_k=3, verbose=True):
    retrieved = hybrid_search(query, top_k=top_k)
    candidates = []
    for doc_id, retr_score in retrieved:
        chunk = corpus[doc_id]
        pred = qa_pipe(question=query, context=chunk["text"])
        candidates.append({
            "country": chunk["country"],
            "answer": pred["answer"],
            "qa_score": pred["score"],
            "retriever_score": retr_score,
            "context_snippet": chunk["text"][:200].replace("\n", " "),
        })
    best = max(candidates, key=lambda c: c["qa_score"])

    if verbose:
        print(f"\nQ: {query}")
        print(f"  → RESPOSTA: {best['answer']}  [{best['country']}, qa_score={best['qa_score']:.3f}]")
        print(f"  Candidatos:")
        for c in candidates:
            print(f"    - [{c['country']}] '{c['answer']}' (qa={c['qa_score']:.3f}, ret={c['retriever_score']:.4f})")

    return best, candidates

In [126]:
# Teste do QA extractivo nas 3 queries
for q in test_queries:
    extractive_qa(q, top_k=3)


Q: What is the capital of Portugal?
  → RESPOSTA: Lisbon  [Portugal, qa_score=0.946]
  Candidatos:
    - [Portugal] 'Lisbon' (qa=0.880, ret=0.0323)
    - [Portugal] 'Lisbon' (qa=0.946, ret=0.0313)
    - [Portugal] 'Assembly of the Republic' (qa=0.007, ret=0.0302)

Q: Which countries border Brazil?
  → RESPOSTA: Uruguay  [Uruguay, qa_score=0.684]
  Candidatos:
    - [Uruguay] 'Uruguay' (qa=0.684, ret=0.0325)
    - [Brazil] 'Ecuador and Chile' (qa=0.222, ret=0.0315)
    - [Brazil] 'Latin America' (qa=0.046, ret=0.0313)

Q: When did Germany reunify?
  → RESPOSTA: World War II  [Luxembourg, qa_score=0.704]
  Candidatos:
    - [Germany] '23 May 1949' (qa=0.328, ret=0.0298)
    - [Luxembourg] 'World War II' (qa=0.704, ret=0.0296)
    - [Cameroon] '1946' (qa=0.069, ret=0.0294)


## 5. QA Abstractivo — RAG com FLAN-T5

**Pipeline:** retriever híbrido (top-3 chunks) → constrói prompt com instrução → FLAN-T5 gera resposta livre.

Diferenças face ao extractivo:
- **Extractivo:** devolve uma *span* exata de um chunk. Bom para factoides, mas só sabe responder se a resposta literal estiver no texto.
- **Abstractivo:** o modelo *gera* a resposta com base no contexto. Consegue parafrasear, sintetizar múltiplas fontes, e dizer "I don't know" quando o contexto não responde.

In [127]:
# Carregar FLAN-T5
from transformers import T5Tokenizer, T5ForConditionalGeneration

T5_MODEL_NAME = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

t5_tokenizer = T5Tokenizer.from_pretrained(T5_MODEL_NAME)
t5_model = T5ForConditionalGeneration.from_pretrained(T5_MODEL_NAME).to(device)
t5_model.eval()

n_params = sum(p.numel() for p in t5_model.parameters()) / 1e6
print(f"FLAN-T5 base carregado: {n_params:.1f}M parâmetros em {device}")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


FLAN-T5 base carregado: 247.6M parâmetros em cuda


In [128]:
import re

def clean_for_prompt(text):
    """Remove citation/footnote markers que sobreviveram ao pré-processamento."""
    text = re.sub(r'\[[\w\d]+\]\.?', '', text)   # [1], [iv], [a], [4]., etc.
    text = re.sub(r'\([a-z]\)\.?', '', text)     # (a), (D), etc.
    text = re.sub(r'\s+', ' ', text)             # normaliza espaços
    return text.strip()

def abstractive_qa(query, top_k=3, max_new_tokens=80, max_chunk_words=200, verbose=True):
    retrieved = hybrid_search(query, top_k=top_k)
    context_parts = []
    countries_used = []
    for doc_id, _ in retrieved:
        chunk = corpus[doc_id]
        words = chunk["text"].split()
        text = " ".join(words[:max_chunk_words]) if len(words) > max_chunk_words else chunk["text"]
        text = clean_for_prompt(text)   # <-- linha nova
        context_parts.append(f"Article about {chunk['country']}: {text}")
        countries_used.append(chunk["country"])
    context = "\n\n".join(context_parts)

    prompt = (
        f"Read the context below and answer the question concisely.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )

    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        outputs = t5_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4, early_stopping=True)
    answer = t5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    has_real_word = bool(re.search(r'[a-zA-Z]{4,}', answer))
    is_short_junk = len(answer) <= 12
    if not has_real_word and is_short_junk:
        answer = "(no clean answer extracted)"

    result = {"query": query, "answer": answer, "context_countries": countries_used,
              "prompt_tokens": inputs.input_ids.shape[1]}
    if verbose:
        print(f"\nQ: {query}")
        print(f"  → RESPOSTA: {answer}")
        print(f"  Contexto: {countries_used}")
    return result

In [129]:
# Teste do QA abstractivo nas 3 queries
for q in test_queries:
    abstractive_qa(q, top_k=3)


Q: What is the capital of Portugal?
  → RESPOSTA: (no clean answer extracted)
  Contexto: ['Portugal', 'Portugal', 'Portugal']

Q: Which countries border Brazil?
  → RESPOSTA: Uruguay to the south
  Contexto: ['Uruguay', 'Brazil', 'Brazil']

Q: When did Germany reunify?
  → RESPOSTA: (no clean answer extracted)
  Contexto: ['Germany', 'Luxembourg', 'Cameroon']


## 6. Demo end-to-end

Compara os dois sistemas QA lado a lado. Inclui:
- **Factoides simples** (capital, língua, população)
- **Paráfrases** (a query usa termos diferentes do texto)
- **Perguntas relacionais** (lista de países, comparações)
- **Pergunta "armadilha"** (informação que o corpus não contém — para ver como cada sistema reage)

In [130]:
def ask(query, top_k=3):
    """Pergunta interativa que mostra as respostas dos dois sistemas QA."""
    print(f"\n{'='*80}")
    print(f"Q: {query}")
    print(f"{'='*80}")

    ext_best, _ = extractive_qa(query, top_k=top_k, verbose=False)
    abs_result = abstractive_qa(query, top_k=top_k, verbose=False)

    print(f"  📌 Extractivo (DistilBERT-SQuAD):  '{ext_best['answer']}'")
    print(f"     [from {ext_best['country']}, qa_score={ext_best['qa_score']:.3f}]")
    print(f"  💬 Abstractivo (FLAN-T5 + RAG):    '{abs_result['answer']}'")
    print(f"     [context: {', '.join(abs_result['context_countries'])}]")

    return ext_best, abs_result

In [131]:
# Demo final — 8 queries cobrindo vários casos
demo_queries = [
    # Factoides simples
    "What is the capital of Portugal?",
    "What is the official language of Italy?",
    # Numéricos
    "What is the population of Iceland?",
    # Relacionais
    "Which countries border Brazil?",
    # Históricos / data específica
    "When did India gain independence?",
    "When did Germany reunify?",
    # Paráfrase (a query usa termo distinto do texto)
    "What currency is used in France?",
    # Armadilha — informação tipicamente não num artigo Wikipedia geral
    "Who won the 2026 FIFA World Cup?",
]

for q in demo_queries:
    ask(q, top_k=3)


Q: What is the capital of Portugal?
  📌 Extractivo (DistilBERT-SQuAD):  'Lisbon'
     [from Portugal, qa_score=0.946]
  💬 Abstractivo (FLAN-T5 + RAG):    '(no clean answer extracted)'
     [context: Portugal, Portugal, Portugal]

Q: What is the official language of Italy?
  📌 Extractivo (DistilBERT-SQuAD):  'Italian'
     [from Italy, qa_score=0.991]
  💬 Abstractivo (FLAN-T5 + RAG):    '(no clean answer extracted)'
     [context: Italy, Monaco, Malta]

Q: What is the population of Iceland?
  📌 Extractivo (DistilBERT-SQuAD):  'two million'
     [from Iceland, qa_score=1.182]
  💬 Abstractivo (FLAN-T5 + RAG):    '(no clean answer extracted)'
     [context: Iceland, Iceland, Iceland]

Q: Which countries border Brazil?
  📌 Extractivo (DistilBERT-SQuAD):  'Uruguay'
     [from Uruguay, qa_score=0.684]
  💬 Abstractivo (FLAN-T5 + RAG):    'Uruguay to the south'
     [context: Uruguay, Brazil, Brazil]

Q: When did India gain independence?
  📌 Extractivo (DistilBERT-SQuAD):  '1947'
     [from Pa

### Notas para o relatório

**Observações típicas:**

1. **Factoides com keyword óbvia** (capital, língua): ambos os sistemas tendem a acertar.
2. **Numéricos** (população): o extractivo costuma devolver a span exata ("376,248"); o abstractivo às vezes generaliza ("around 376,000").
3. **Relacionais** (países que fazem fronteira): o extractivo só extrai uma única span, perdendo a lista completa; o abstractivo consegue enumerar os vários países.
4. **Paráfrases** (currency vs euro/dollar): o BM25 falhava aqui; o retriever semântico + qualquer dos dois QA resolvem.
5. **Armadilha** (Mundial 2026): o corpus não tem essa informação. O abstractivo idealmente diz "I don't know"; o extractivo nunca diz isso — vai sempre devolver alguma span, frequentemente errada (limitação conhecida dos modelos extractivos).

**Conclusão do pipeline:** o sistema híbrido (retriever + ambos os QA) é robusto e cada componente compensa as limitações dos outros.